In [1]:
import warnings
from tqdm import TqdmWarning
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=TqdmWarning)
warnings.filterwarnings("ignore", message="Detected IPython.*")

import pandas as pd
from pathlib import Path
import re
import os
import json
import numpy as np
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import pandas as pd
import networkx as nx
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
import utils as ut
import model_wrappers as mw
from hyperparameter import model_configs


current_path = Path(__file__).resolve().parent if '__file__' in globals() else Path().resolve()

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


In [2]:
import os
from pathlib import Path

# Assuming current_path is already defined as in your configuration
base_path = current_path.parent / "data" / "silver" / "evaluation_benchmark"

# List all subdirectories in base_path
folders = [f for f in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, f))]

# Create a dictionary with folder name as key and full path as value
folder_dict = {f: base_path / f for f in folders}

# Model training

In [3]:
results_dfs = []

for folder_name, folder_path in folder_dict.items():
    print(f"\nProcessing dataset: {folder_name}")
    symbolic_equations = {"causal": {}, "traditional": {}}
    trained_models = {}
    results = []

    # Paths to experimental data
    train_path = folder_path / "train.csv"
    test_path = folder_path / "test.csv"
    intervention_path = folder_path / "interventions.json"
    adj_matrix_path = folder_path / "adj_matrix.csv"
    # --- Intervention scenario ---
    with open(intervention_path, "r") as f:
        interventions = json.load(f)["environments"]

    # Load train/test data without headers to determine column names
    temp_train_df = pd.read_csv(train_path, header=None)
    n_columns = temp_train_df.shape[1]
    column_names = [f"X{i}" for i in range(n_columns)]
    target_column = 'X' + str(interventions[0]['effect_idxs'][0]-1)

    # Reload with column names
    train_df = pd.read_csv(train_path, header=None, names=column_names)
    test_df = pd.read_csv(test_path, header=None, names=column_names)

    X_train = train_df.drop(columns=[target_column])
    y_train = train_df[target_column]
    X_test = test_df.drop(columns=[target_column])
    y_test = test_df[target_column]

    # Combine reference + test into one dataset
    combined_data = []
    for env in interventions:
        test_data = pd.DataFrame(env["test_data"], columns=column_names)
        ref_data = pd.DataFrame(env["reference_data"], columns=column_names)
        combined = pd.concat([test_data, ref_data], ignore_index=True)
        combined_data.append(combined)

    # Evaluate on each combined intervention dataset
    inter_df = pd.concat(combined_data)
    X_int = inter_df.drop(columns=[target_column])
    y_int = inter_df[target_column]


    # Load causal graph adjacency matrix
    adj_matrix = pd.read_csv(adj_matrix_path, header=None).values

    # Build graph G from adjacency matrix
    import networkx as nx
    G = nx.DiGraph()
    for i, src in enumerate(column_names):
        for j, tgt in enumerate(column_names):
            if adj_matrix[i, j] == 1:
                G.add_edge(src, tgt)

    # Train and evaluate traditional and causal
    for name, cfg in model_configs.items():
        ModelClass = cfg["model_class"]
        params = cfg.get("params", {})
        param_grid = cfg.get("param_grid", None)

        # --- Causal version ---
        causal_models = ut.train_causal_models(train_df, G, ModelClass,
                                            model_params=params, param_grid=param_grid)
        causal_preds = ut.predict_causal(test_df, G, causal_models, what_if=False)
        rmse_c, wape_c, mae_c = ut.evaluate_metrics(test_df[target_column], causal_preds[target_column])
        results.append(["Causal", name, rmse_c, wape_c, mae_c, "Test"])

        # Extract equations for causal models (SymbolicRegression only)
        if name == "SymbolicRegression":
            symbolic_equations["causal"][name] = {}
            for node, model in causal_models.items():
                try:
                    equation = str(model.model.get_best()['equation'])
                    equation = ut.round_numbers_in_string(equation)
                    symbolic_equations["causal"][name][node] = equation
                    print(f"[Causal {name}] Node '{node}': {equation}")
                except:
                    symbolic_equations["causal"][name][node] = "No equation available"

        # --- Traditional version ---
        if param_grid:
            base_model = ModelClass(**params)
            grid_search = GridSearchCV(base_model, param_grid=param_grid,
                                    cv=4, n_jobs=-1, scoring='neg_mean_absolute_error')
            grid_search.fit(X_train, y_train)
            model = grid_search.best_estimator_
            print(f"[GridSearchCV] Best params for Traditional '{name}': {grid_search.best_params_}")
        else:
            model = ModelClass(**params)
            model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        rmse_t, wape_t, mae_t = ut.evaluate_metrics(y_test, y_pred, normalize=True)
        results.append(["Traditional", name, rmse_t, wape_t, mae_t, "Test"])

        if name == "SymbolicRegression":
            try:
                equation = str(model.model.get_best()['equation'])
                equation = ut.round_numbers_in_string(equation)
                symbolic_equations["traditional"][name] = equation
                print(f"[Traditional {name}] Target equation: {equation}")
            except:
                symbolic_equations["traditional"][name] = "No equation available"

        # Evaluate interventions # 
        # Causal predictions
        causal_preds_int = ut.predict_causal(inter_df, G, causal_models, what_if=True)
        rmse_c_int, wape_c_int, mae_c_int = ut.evaluate_metrics(y_int, causal_preds_int[target_column], normalize=True)
        results.append(["Causal", name, rmse_c_int, wape_c_int, mae_c_int, f"Intervention"])

        # Traditional predictions
        y_pred_int = model.predict(X_int)
        rmse_t_int, wape_t_int, mae_t_int = ut.evaluate_metrics(y_int, y_pred_int, normalize=True)
        results.append(["Traditional", name, rmse_t_int, wape_t_int, mae_t_int, f"Intervention"])

        # Store trained models
        trained_models[name] = {"causal": causal_models, "traditional": model}


    # Results table
    results_df = pd.DataFrame(results, columns=["Model_Type", "Algorithm", "RMSE", "WAPE", "MAE", "Task"])
    results_df = results_df[["Model_Type", "Algorithm", "MAE", "RMSE", "WAPE", "Task"]]
    results_df['experiment'] = folder_name

    results_dfs.append(results_df)

    # Display symbolic equations
    print("\n" + "="*80)
    print("SYMBOLIC REGRESSION EQUATIONS")
    print("="*80)

    if symbolic_equations["causal"]:
        print("\n--- CAUSAL MODELS ---")
        for model_name, node_equations in symbolic_equations["causal"].items():
            print(f"\n{model_name}:")
            for node, equation in node_equations.items():
                print(f"  {node}: {equation}")

    if symbolic_equations["traditional"]:
        print("\n--- TRADITIONAL MODELS ---")
        for model_name, equation in symbolic_equations["traditional"].items():
            print(f"{model_name}: {equation}")

    print("\n" + "="*80)

    # Display results
    print(results_df)

    # Return trained models
    trained_models


Processing dataset: csuite_nonlin_simpson
[GridSearchCV] Best params for Traditional 'RandomForest': {'max_depth': 10, 'min_samples_split': 15, 'n_estimators': 1000}
[GridSearchCV] Best params for Traditional 'MLP': {'alpha': 0.01, 'hidden_layer_sizes': (100, 50), 'learning_rate': 'constant'}
[GridSearchCV] Best params for Traditional 'XGBoost': {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 1000}
Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython
Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython
Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython
Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython
Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython
Detected IPython. L

In [4]:
all_results_df = pd.concat(results_dfs, ignore_index=True)

# Group by Model_Type, Algorithm, and Task, and take the median of the numeric columns
aggregated_results_df = all_results_df.groupby(['Model_Type', 'Algorithm', 'Task']).agg({
    'MAE': 'median',
    'RMSE': 'median',
    'WAPE': 'median',
    
}).reset_index()


aggregated_results_df

,Model_Type,Algorithm,Task,MAE,RMSE,WAPE
0,Causal,GAM,Intervention,1.045307,1.371046,104.530741
1,Causal,GAM,Test,0.181897,0.232505,23.129592
2,Causal,LinearRegression,Intervention,1.085090,1.318203,108.508979
3,Causal,LinearRegression,Test,0.276197,0.353732,33.366032
4,Causal,MLP,Intervention,1.048018,1.378904,104.801797
5,Causal,MLP,Test,0.181891,0.232345,23.140901
6,Causal,RandomForest,Intervention,1.052443,1.368126,105.244270
7,Causal,RandomForest,Test,0.189632,0.247017,24.109542
8,Causal,SymbolicRegression,Intervention,1.085327,1.352616,108.532698
9,Causal,SymbolicRegression,Test,0.195085,0.251543,24.989493


In [5]:
df = aggregated_results_df.copy()

df = df.round(4)

df['Model_Type'] = df['Model_Type'].replace({'Causal': 'CML', 'Traditional': 'ML'})
df['Algorithm'] = df['Algorithm'].replace({'RandomForest': 'RForest', 'SymbolicRegression': 'Symbolic',
                                           'XGBRegression': 'XGB', 'LinearRegression': 'Linreg',})

df.columns = ["Type", "Model", "Task", "MAE", "RMSE", "WAPE"]

# Filter for Intervention task and drop Task column
intervention_df = df[df["Task"] == "Intervention"].drop(columns=["Task"]).sort_values(by=["MAE"])

# Filter for Test task and drop Task column
test_df = df[df["Task"] == "Test"].drop(columns=["Task"]).sort_values(by=["MAE"])

# Display the Intervention table with 5 significant digits for numeric columns
print("Intervention Results:")
intervention_df = intervention_df.style.hide(axis='index').format({'MAE': '{:.5g}', 'RMSE': '{:.5g}', 'WAPE': '{:.5g}'})
display(intervention_df)

# Display the Test table with 5 significant digits for numeric columns
print("\nTest Results:")
display(test_df.style.hide(axis='index').format({'MAE': '{:.5g}', 'RMSE': '{:.5g}', 'WAPE': '{:.5g}'}))

Intervention Results:


Type,Model,MAE,RMSE,WAPE
CML,GAM,1.0453,1.371,104.53
CML,XGBoost,1.047,1.3836,104.7
CML,MLP,1.048,1.3789,104.8
CML,RForest,1.0524,1.3681,105.24
CML,Linreg,1.0851,1.3182,108.51
CML,Symbolic,1.0853,1.3526,108.53
ML,GAM,1.1267,1.4374,112.67
ML,XGBoost,1.127,1.4363,112.7
ML,RForest,1.1343,1.444,113.43
ML,MLP,1.1366,1.4545,113.66



Test Results:


Type,Model,MAE,RMSE,WAPE
CML,GAM,0.1819,0.2325,23.13
CML,MLP,0.1819,0.2323,23.141
CML,XGBoost,0.1852,0.2368,23.582
CML,RForest,0.1896,0.247,24.11
ML,GAM,0.1908,0.2388,24.458
ML,MLP,0.1914,0.2403,24.549
ML,XGBoost,0.1919,0.2412,24.604
ML,RForest,0.1933,0.243,24.78
CML,Symbolic,0.1951,0.2515,24.989
ML,Symbolic,0.2145,0.2722,27.508


# Individual results for the datasets

In [6]:
all_results_df = pd.concat(results_dfs, ignore_index=True)

for experiment in all_results_df['experiment'].unique():

    print(experiment)
    df = all_results_df[all_results_df['experiment'] == experiment]

    df = df.round(4)

    df['Model_Type'] = df['Model_Type'].replace({'Causal': 'CML', 'Traditional': 'ML'})
    df['Algorithm'] = df['Algorithm'].replace({'RandomForest': 'RForest', 'SymbolicRegression': 'Symbolic',
                                            'XGBRegression': 'XGB', 'LinearRegression': 'Linreg',})

    df.columns = ["Type", "Model",  "MAE", "RMSE", "WAPE", "Task", "experiment"]

    # Filter for Intervention task and drop Task column
    intervention_df = df[df["Task"] == "Intervention"].drop(columns=["Task"]).sort_values(by=["MAE"])

    # Filter for Test task and drop Task column
    test_df = df[df["Task"] == "Test"].drop(columns=["Task"]).sort_values(by=["MAE"])

    # Display the Intervention table with 5 significant digits for numeric columns
    print("Intervention Results:")
    intervention_df = intervention_df.style.hide(axis='index').format({'MAE': '{:.5g}', 'RMSE': '{:.5g}', 'WAPE': '{:.5g}'})
    display(intervention_df)

    # Display the Test table with 5 significant digits for numeric columns
    print("\nTest Results:")
    display(test_df.style.hide(axis='index').format({'MAE': '{:.5g}', 'RMSE': '{:.5g}', 'WAPE': '{:.5g}'}))
    

csuite_nonlin_simpson
Intervention Results:


Type,Model,MAE,RMSE,WAPE,experiment
ML,MLP,0.8487,1.0594,84.87,csuite_nonlin_simpson
ML,XGBoost,0.8668,1.0874,86.678,csuite_nonlin_simpson
ML,RForest,0.8743,1.0913,87.435,csuite_nonlin_simpson
ML,Linreg,0.8825,1.0662,88.25,csuite_nonlin_simpson
ML,GAM,0.8979,1.1179,89.787,csuite_nonlin_simpson
CML,MLP,0.9182,1.1464,91.82,csuite_nonlin_simpson
CML,XGBoost,0.9185,1.1472,91.852,csuite_nonlin_simpson
CML,GAM,0.9189,1.1472,91.887,csuite_nonlin_simpson
CML,RForest,0.9196,1.1479,91.957,csuite_nonlin_simpson
CML,Symbolic,0.9221,1.1442,92.206,csuite_nonlin_simpson



Test Results:


Type,Model,MAE,RMSE,WAPE,experiment
CML,GAM,0.1216,0.1527,20.515,csuite_nonlin_simpson
CML,MLP,0.122,0.153,20.579,csuite_nonlin_simpson
CML,Symbolic,0.1237,0.155,20.876,csuite_nonlin_simpson
CML,XGBoost,0.1248,0.1566,21.055,csuite_nonlin_simpson
CML,RForest,0.1266,0.1597,21.367,csuite_nonlin_simpson
CML,Linreg,0.1533,0.1995,25.861,csuite_nonlin_simpson
ML,MLP,0.162,0.2049,20.259,csuite_nonlin_simpson
ML,GAM,0.1629,0.2066,20.382,csuite_nonlin_simpson
ML,XGBoost,0.1639,0.2083,20.505,csuite_nonlin_simpson
ML,RForest,0.1656,0.2107,20.719,csuite_nonlin_simpson


csuite_large_backdoor
Intervention Results:


...

Type,Model,MAE,RMSE,WAPE,experiment
ML,GAM,0.2186,0.2711,28.533,csuite_weak_arrows
ML,XGBoost,0.2199,0.2742,28.703,csuite_weak_arrows
ML,MLP,0.2209,0.2757,28.84,csuite_weak_arrows
ML,RForest,0.2209,0.2752,28.841,csuite_weak_arrows
ML,Linreg,0.238,0.3002,31.076,csuite_weak_arrows
CML,MLP,0.2418,0.3006,25.702,csuite_weak_arrows
CML,GAM,0.2422,0.3013,25.745,csuite_weak_arrows
CML,XGBoost,0.2457,0.3057,26.109,csuite_weak_arrows
ML,Symbolic,0.2498,0.3186,32.613,csuite_weak_arrows
CML,RForest,0.2526,0.3152,26.852,csuite_weak_arrows


csuite_symprod_simpson
Intervention Results:


Type,Model,MAE,RMSE,WAPE,experiment
ML,RForest,0.995,1.3191,99.501,csuite_symprod_simpson
ML,XGBoost,0.9988,1.3266,99.883,csuite_symprod_simpson
CML,GAM,0.9996,1.3305,99.96,csuite_symprod_simpson
CML,MLP,0.9999,1.3294,99.993,csuite_symprod_simpson
ML,GAM,1.0001,1.3297,100.01,csuite_symprod_simpson
CML,XGBoost,1.0002,1.3304,100.02,csuite_symprod_simpson
CML,RForest,1.0012,1.3312,100.12,csuite_symprod_simpson
CML,Symbolic,1.008,1.3342,100.8,csuite_symprod_simpson
ML,Symbolic,1.008,1.3342,100.8,csuite_symprod_simpson
ML,MLP,1.0258,1.3671,102.58,csuite_symprod_simpson



Test Results:


Type,Model,MAE,RMSE,WAPE,experiment
ML,GAM,0.0712,0.1039,7.692,csuite_symprod_simpson
ML,XGBoost,0.0736,0.1055,7.947,csuite_symprod_simpson
ML,MLP,0.0758,0.1084,8.189,csuite_symprod_simpson
ML,RForest,0.0775,0.1094,8.3739,csuite_symprod_simpson
ML,Symbolic,0.0884,0.1207,9.5415,csuite_symprod_simpson
CML,GAM,0.1121,0.1637,7.6174,csuite_symprod_simpson
CML,MLP,0.1128,0.1641,7.663,csuite_symprod_simpson
CML,XGBoost,0.1169,0.1678,7.9469,csuite_symprod_simpson
CML,RForest,0.1255,0.1788,8.5312,csuite_symprod_simpson
CML,Symbolic,0.1403,0.1918,9.5365,csuite_symprod_simpson
